In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import load_config
from src.audit.run_audit import run
from src.features.value_surface import savitzky_golay, vxy
from src.data_io.load import load_match
from src.data_io.transform import build_normalised

pd.set_option("display.width", 160)
cfg = load_config(ROOT / "config.yaml")

In [2]:
res_end = run(cfg=cfg, verbose=False)  # loops every match in matches_dir, one in RAM at a time
res_end.save_csv()

In [3]:
res_end.alignment.event_type.value_counts()  # now carries player_possession AND passing_option checks


event_type
passing_option       48276
player_possession    18651
Name: count, dtype: int64

In [4]:
# V(x, y): xThreat value surface from all passing_option events, three interchangeable methods
surfaces = {m: vxy(res_end.alignment, method=m, bin_m=cfg["audit"]["heatmap_bin_m"]) for m in ("grid", "kde", "spline")}
for m, v in surfaces.items():
    print(m, "V range:", v.V.min(), v.V.max(), "| sampled cells:", (v.n > 0).sum(), "/", len(v))
surfaces["grid"].head()


grid V range: 3.9999999999999996e-05 0.16895238095238096 | sampled cells: 292 / 294
kde V range: 4.521767665078402e-05 0.13592728670096857 | sampled cells: 292 / 294
spline V range: -0.031159670256053054 0.1251177773862027 | sampled cells: 292 / 294


,x,y,V,n,method
0,-50.0,-31.5,0.000233,3.0,grid
1,-50.0,-26.5,0.000175,4.0,grid
2,-50.0,-21.5,0.000367,6.0,grid
3,-50.0,-16.5,0.000158,19.0,grid
4,-50.0,-11.5,0.000077,44.0,grid


In [5]:
# Fully separate feature: SG-smoothed velocities, gap-aware over detected-only runs.
# Needs one match's raw tracking (not the audit's concatenated alignment table).

m0 = load_match(cfg["data"]["matches_dir"], 1874553)
_, tracking0 = build_normalised(m0)
sg = savitzky_golay(tracking0, fps=cfg["data"]["fps"])
sg[["vx", "vy", "speed"]].describe()


,vx,vy,speed
count,471512.000000,471512.000000,471512.000000
mean,-0.234866,0.017422,5.306016
std,14.216431,9.548922,16.284668
min,-394.693290,-295.174978,0.001667
25%,-0.810000,-1.256667,1.469970
50%,0.573333,0.030000,2.442252
75%,2.105000,1.296667,3.683631
max,433.750584,299.747532,511.935757
